# 1차전처리

In [14]:
import os
import json
import shutil
from pathlib import Path
import yaml
import random
from tqdm import tqdm
import math # math 모듈 임포트 추가

# 1. 경로 설정
origin_root = Path("origin_sample_dataset") # <- 여기에 원본데이터셋 경로설정
yolo_root = Path("YOLOv11Dataset") # <- 여기에 Yolo v11 변환데이터셋 경로설정
yolo_root.mkdir(parents=True, exist_ok=True)

# 2. YOLO 디렉토리 구조 생성 (labels만 생성, images는 사용자가 별도로 관리)
dirs = [
    "labels/train",
    "labels/val",
    "labels/test"
]

for d in dirs:
    (yolo_root / d).mkdir(parents=True, exist_ok=True)

# 3. 클래스 매핑 정보 생성
class_mapping = {
    ("01", 0): 0, # '01' (배) + disease 0 = 0: '배 정상'
    ("01", 1): 1, # '01' (배) + disease 1 = 1: '배검은별무늬병'
    ("01", 2): 2, # '01' (배) + disease 2 = 2: '배과수화상병'

    ("02", 0): 8, # '02' (사과) + disease 0 = 8: '사과 정상'
    ("02", 3): 3, # '02' (사과) + disease 1 = 3: '사과갈색무늬병'
    ("02", 4): 4, # '02' (사과) + disease 2 = 4: '사과과수화상병'
    ("02", 5): 5, # '02' (사과) + disease 3 = 5: '사과부란병'
    ("02", 6): 6, # '02' (사과) + disease 4 = 6: '사과점무늬낙엽병'
    ("02", 7): 7  # '02' (사과) + disease 5 = 7: '사과탄저병'
}

# JSON → YOLO 레이블 변환 함수
def convert_label(json_path, img_filename_stem, img_width, img_height):
    """
    JSON 파일을 읽어 YOLO 형식의 텍스트 라벨을 반환합니다.
    바운딩 박스 좌표가 이미지 영역을 벗어나는 경우 (음수 또는 이미지 크기 초과)
    해당 좌표를 0 또는 이미지 크기로 클리핑합니다.
    Args:
        json_path (Path): JSON 파일의 전체 경로.
        img_filename_stem (str): 이미지 파일명 (확장자 제외), JSON 파일명과 동일하다고 가정.
        img_width (int): 원본 이미지의 너비.
        img_height (int): 원본 이미지의 높이.
    Returns:
        str: YOLO 형식의 라벨 문자열 (각 라인마다 하나의 객체), 또는 변환 실패 시 빈 문자열.
    """
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        # print(f"  ! Error reading JSON file {json_path}: {e}. Skipping.")
        return ""
    
    # 1. 파일명에서 작물 정보 추출
    filename_parts = img_filename_stem.split('_')
    if len(filename_parts) > 4:
        crop_code = filename_parts[4] # '01' (배) 또는 '02' (사과)
    else:
        # print(f"    ! Warning: Could not extract crop code from filename: {img_filename_stem}. Skipping.")
        return "" # 유효하지 않은 파일명이면 빈 문자열 반환

    # 2. JSON에서 질병 정보 추출
    try:
        json_disease_value = data['annotations']['disease']
    except KeyError:
        # print(f"    ! Error: Missing 'annotations.disease' key in JSON file {json_path.name}. Skipping.")
        return ""

    # 3. class_mapping을 사용하여 최종 class_id 결정
    class_key = (crop_code, json_disease_value)
    class_id = class_mapping.get(class_key) # 매핑에 없으면 None 반환
    
    if class_id is None:
        # print(f"    ! Warning: No class mapping for {class_key} (from {json_path.name}). Skipping.")
        return "" # 매핑 정보가 없으면 빈 문자열 반환

    # 바운딩 박스 처리
    bbox_lines = []
    # JSON 파일에 'points' 키가 없을 경우를 대비하여 예외 처리 추가 (기존 코드에 없던 부분)
    if 'annotations' not in data or 'points' not in data['annotations']:
        # print(f"    ! Warning: Missing 'annotations' or 'points' in JSON file {json_path.name}. Skipping this file.")
        return ""

    for point in data['annotations']['points']:
        xtl = point['xtl']
        ytl = point['ytl']
        xbr = point['xbr']
        ybr = point['ybr']
        
        # 바운딩 박스 좌표 클리핑 (0과 이미지 크기 사이로 조정)
        xtl_clipped = max(0, xtl)
        ytl_clipped = max(0, ytl)
        xbr_clipped = min(img_width, xbr)
        ybr_clipped = min(img_height, ybr)

        # 클리핑 후 유효한 바운딩 박스인지 확인 (너비나 높이가 0 이하인 경우 제외)
        current_width = xbr_clipped - xtl_clipped
        current_height = ybr_clipped - ytl_clipped

        if current_width <= 0 or current_height <= 0:
            # print(f"    ! Warning: Clipped bbox for {json_path.name} has zero or negative dimension ({current_width}x{current_height}). Skipping this bbox.")
            continue # 유효하지 않은 바운딩 박스는 스킵

        # YOLO 형식으로 정규화 (x_center y_center width height)
        x_center = ((xtl_clipped + xbr_clipped) / 2) / img_width
        y_center = ((ytl_clipped + ybr_clipped) / 2) / img_height
        width = current_width / img_width
        height = current_height / img_height
        
        # 정규화된 값이 0-1 범위를 벗어나는 극단적인 경우는 발생하지 않아야 하지만,
        # 만약을 위해 다시 클리핑 (부동 소수점 오차 방지)
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        width = max(0.0, min(1.0, width))
        height = max(0.0, min(1.0, height))

        bbox_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return "\n".join(bbox_lines)

# 개별 JSON 파일 처리 함수
def process_json(json_file: Path, target_type: str):
    """
    단일 JSON 파일을 읽어 YOLO .txt 라벨 파일을 생성합니다.
    이미 해당 라벨 파일이 존재하면 건너뜁니다.
    Args:
        json_file (Path): 처리할 JSON 파일의 Path 객체.
        target_type (str): 'train', 'val', 'test' 중 하나.
    """
    # .jpg, .JPG, .png, .PNG 확장자가 붙어있는 경우 제거 (파일명 스템만 사용)
    img_filename_stem = json_file.stem.replace(".jpg", "").replace(".JPG", "").replace(".png", "").replace(".PNG", "")

    # YOLO 형식 라벨이 저장될 최종 경로
    label_dir = yolo_root / "labels" / target_type
    txt_path = label_dir / f"{img_filename_stem}.txt" 
    
    # --- 핵심 변경 부분: 파일이 이미 존재하면 건너뛰기 ---
    if txt_path.exists():
        # print(f"  [SKIP] 라벨 파일이 이미 존재합니다: {txt_path.name}")
        return # 파일이 존재하면 이 함수 호출을 여기서 종료하여 건너뜁니다.
    # ----------------------------------------------------

    # JSON에서 이미지 크기 추출 (description 필드 사용)
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        
        img_width = json_data['description']['width']
        img_height = json_data['description']['height']
    except KeyError as e:
        # print(f"    ! Error: Missing key {e} in JSON file {json_file}. Skipping.")
        return
    except Exception as e:
        # print(f"    ! Error reading JSON for {json_file}: {e}. Skipping.")
        return

    # 레이블 변환
    yolo_label = convert_label(json_file, img_filename_stem, img_width, img_height)
    
    # 변환된 라벨이 없으면 (예: 매핑 실패) 파일 생성 건너뛰기
    if not yolo_label:
        return

    # YOLO 형식 라벨 저장 (이제 이 부분은 파일이 없을 때만 실행됩니다)
    label_dir.mkdir(parents=True, exist_ok=True) # 폴더가 없을 경우 대비
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(yolo_label)

# 데이터셋 처리 함수 (tqdm 적용)
def process_dataset(src_type: str):
    """
    원본 데이터셋의 Training/Validation 폴더를 탐색하여 JSON 파일을 처리하고
    YOLO 형식의 .txt 라벨 파일을 생성합니다.
    Args:
        src_type (str): 'Training' 또는 'Validation'.
    """
    print(f"\nProcessing {src_type} data...")
    
    for folder in (origin_root / src_type).iterdir():
        # '[라벨]'로 시작하는 폴더만 처리
        if folder.name.startswith("[라벨]"):
            json_files = list(folder.glob("*.json"))
            random.seed(42) # 시드 고정
            random.shuffle(json_files) # 파일 목록 섞기

            if src_type == "Training":
                target_type = "train"
                print(f"  Converting JSONs from '{folder.name}' to '{target_type}' labels...")
                for json_file in tqdm(json_files, desc=f"  {target_type.upper()} labels"):
                    process_json(json_file, target_type)
            elif src_type == "Validation":
                # Validation 데이터는 val과 test로 5:5 분할
                split_idx = math.ceil(len(json_files) * 0.5) # 절반(올림)을 val로
                val_files = json_files[:split_idx]
                test_files = json_files[split_idx:]
                
                print(f"  Total JSONs in '{folder.name}': {len(json_files)}")
                print(f"  Allocating {len(val_files)} to 'val' and {len(test_files)} to 'test'.")

                print(f"  Converting JSONs from '{folder.name}' to 'val' labels...")
                for json_file in tqdm(val_files, desc="  VAL labels"):
                    process_json(json_file, "val")
                
                print(f"  Converting JSONs from '{folder.name}' to 'test' labels...")
                for json_file in tqdm(test_files, desc="  TEST labels"):
                    process_json(json_file, "test")

# 6. 데이터 처리 실행
print("--- Starting main dataset conversion ---")
process_dataset("Training")
process_dataset("Validation")
print("--- Main dataset conversion finished ---")

# 7. dataset.yaml 파일 생성
yaml_content = {
    'path': str(yolo_root.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',

    'names': {
        0: '배 정상',
        1: '배검은별무늬병',
        2: '배과수화상병',
        3: '사과갈색무늬병',
        4: '사과과수화상병',
        5: '사과부란병',
        6: '사과점무늬낙엽병',
        7: '사과탄저병',
        8: '사과 정상'
    }
}

# dataset.yaml은 항상 최신 정보로 덮어쓰는 것이 좋습니다.
with open(yolo_root / "dataset.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, allow_unicode=True, sort_keys=False)

# 8. 분할 결과 통계 출력
def print_stats():
    print("\n---")
    print("Dataset Split Statistics (Labels Only):")
    for split in ['train', 'val', 'test']:
        label_count = len(list((yolo_root / "labels" / split).glob("*.txt")))
        print(f"  - {split}: {label_count} labels")
    print("---\n")

print("\nJSON to YOLO label conversion completed successfully!")
print(f"YOLO label files are generated in: {yolo_root / 'labels'}")
print(f"The 'dataset.yaml' file is located at: {yolo_root / 'dataset.yaml'}")
print_stats()

print("Important: Remember to correctly configure 'train', 'val', and 'test' paths in 'dataset.yaml'")
print("to point to your actual image directories for YOLOv11 training.")

--- Starting main dataset conversion ---

Processing Training data...
  Converting JSONs from '[라벨]배_0.정상' to 'train' labels...


  TRAIN labels: 100%|██████████| 20434/20434 [00:01<00:00, 20326.01it/s]


  Converting JSONs from '[라벨]배_1.질병' to 'train' labels...


  TRAIN labels: 100%|██████████| 2557/2557 [00:00<00:00, 20505.79it/s]


  Converting JSONs from '[라벨]사과_0.정상' to 'train' labels...


...

  Total JSONs in '[라벨]사과_1.질병': 1036
  Allocating 518 to 'val' and 518 to 'test'.
  Converting JSONs from '[라벨]사과_1.질병' to 'val' labels...


  VAL labels: 100%|██████████| 518/518 [00:00<00:00, 1057.81it/s]


  Converting JSONs from '[라벨]사과_1.질병' to 'test' labels...


  TEST labels: 100%|██████████| 518/518 [00:00<00:00, 985.55it/s] 


--- Main dataset conversion finished ---

JSON to YOLO label conversion completed successfully!
YOLO label files are generated in: YOLOv11Dataset\labels
The 'dataset.yaml' file is located at: YOLOv11Dataset\dataset.yaml

---
Dataset Split Statistics (Labels Only):
  - train: 60015 labels
  - val: 3753 labels
  - test: 3754 labels
---

Important: Remember to correctly configure 'train', 'val', and 'test' paths in 'dataset.yaml'
to point to your actual image directories for YOLOv11 training.


In [15]:
import os
import json
from pathlib import Path
import random
from tqdm import tqdm
import math

# 1. 경로 설정 (메인 스크립트와 동일해야 함)
origin_root = Path("origin_sample_dataset") # <- 여기에 원본데이터셋 경로설정
yolo_root = Path("YOLOv11Dataset") # <- 여기에 Yolo v11 변환데이터셋 경로설정
yolo_root.mkdir(parents=True, exist_ok=True) # 폴더가 없으면 생성

# 2. YOLO 디렉토리 구조 생성 (labels만 생성, 이미 메인 스크립트에서 생성되었을 것임)
dirs = [
    "labels/train",
    "labels/val",
    "labels/test"
]

for d in dirs:
    (yolo_root / d).mkdir(parents=True, exist_ok=True)

# 3. 클래스 매핑 정보 (메인 스크립트와 동일해야 함)
class_mapping = {
    ("01", 0): 0, # '01' (배) + disease 0 = 0: '배 정상'
    ("01", 1): 1, # '01' (배) + disease 1 = 1: '배검은별무늬병'
    ("01", 2): 2, # '01' (배) + disease 2 = 2: '배과수화상병'

    ("02", 0): 8, # '02' (사과) + disease 0 = 8: '사과 정상'
    ("02", 3): 3, # '02' (사과) + disease 1 = 3: '사과갈색무늬병'
    ("02", 4): 4, # '02' (사과) + disease 2 = 4: '사과과수화상병'
    ("02", 5): 5, # '02' (사과) + disease 3 = 5: '사과부란병'
    ("02", 6): 6, # '02' (사과) + disease 4 = 6: '사과점무늬낙엽병'
    ("02", 7): 7  # '02' (사과) + disease 5 = 7: '사과탄저병'
}

def convert_label_from_filename_override(json_path, img_filename_stem, img_width, img_height):
    """
    JSON 파일을 읽어 YOLO 형식의 텍스트 라벨을 반환합니다.
    이 함수는 JSON 내부의 'disease' 값을 무시하고, 파일명에서 'crop_code'와 'disease_code'를 추출하여
    클래스 ID를 결정합니다. 바운딩 박스 좌표 클리핑 로직 포함.
    Args:
        json_path (Path): JSON 파일의 전체 경로.
        img_filename_stem (str): 이미지 파일명 (확장자 제외), JSON 파일명과 동일하다고 가정.
        img_width (int): 원본 이미지의 너비.
        img_height (int): 원본 이미지의 높이.
    Returns:
        str: YOLO 형식의 라벨 문자열 (각 라인마다 하나의 객체), 또는 변환 실패 시 빈 문자열.
    """
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        # print(f"  ! Error reading JSON file {json_path}: {e}. Skipping.")
        return ""
    
    # 파일명에서 작물 정보와 질병 정보 추출 (새로운 로직)
    # 예: V006_80_1_07_02_01_23_2_6319b_20201022_18 -> '02' (사과), '07' (사과탄저병)
    filename_parts = img_filename_stem.split('_')
    
    if len(filename_parts) >= 5: # 최소한 5번째 인덱스까지 있어야 함 (0-indexed)
        # crop_code: _ 기준으로 5번째 (인덱스 4)
        extracted_crop_code = filename_parts[4]
        # disease_code: _ 기준으로 4번째 (인덱스 3) -> 이를 정수형으로 변환
        try:
            extracted_disease_code = int(filename_parts[3])
        except ValueError:
            # print(f"    ! Warning: Could not parse disease code from filename part '{filename_parts[3]}' in {img_filename_stem}. Skipping.")
            return ""
    else:
        # print(f"    ! Warning: Filename '{img_filename_stem}' does not have enough parts for extraction. Skipping.")
        return ""

    # class_mapping을 사용하여 최종 class_id 결정 (추출된 crop_code와 disease_code 사용)
    class_key = (extracted_crop_code, extracted_disease_code)
    class_id = class_mapping.get(class_key)
    
    if class_id is None:
        # print(f"    ! Warning: No class mapping for {class_key} (extracted from filename: crop='{extracted_crop_code}', disease={extracted_disease_code}) from {json_path.name}. Skipping.")
        return ""

    # 바운딩 박스 처리 (메인 스크립트와 동일한 클리핑 로직 적용)
    bbox_lines = []
    # JSON 파일에 'points' 키가 없을 경우를 대비하여 예외 처리
    if 'annotations' not in data or 'points' not in data['annotations']:
        # print(f"    ! Warning: Missing 'annotations' or 'points' in JSON file {json_path.name}. Skipping this file.")
        return ""

    for point in data['annotations']['points']:
        xtl = point['xtl']
        ytl = point['ytl']
        xbr = point['xbr']
        ybr = point['ybr']
        
        # 바운딩 박스 좌표 클리핑 (0과 이미지 크기 사이로 조정)
        xtl_clipped = max(0, xtl)
        ytl_clipped = max(0, ytl)
        xbr_clipped = min(img_width, xbr)
        ybr_clipped = min(img_height, ybr)

        # 클리핑 후 유효한 바운딩 박스인지 확인 (너비나 높이가 0 이하인 경우 제외)
        current_width = xbr_clipped - xtl_clipped
        current_height = ybr_clipped - ytl_clipped

        if current_width <= 0 or current_height <= 0:
            # print(f"    ! Warning: Clipped bbox for {json_path.name} has zero or negative dimension ({current_width}x{current_height}). Skipping this bbox.")
            continue # 유효하지 않은 바운딩 박스는 스킵

        # YOLO 형식으로 정규화 (x_center y_center width height)
        x_center = ((xtl_clipped + xbr_clipped) / 2) / img_width
        y_center = ((ytl_clipped + ybr_clipped) / 2) / img_height
        width = current_width / img_width
        height = current_height / img_height
        
        # 정규화된 값이 0-1 범위를 벗어나는 극단적인 경우는 발생하지 않아야 하지만,
        # 만약을 위해 다시 클리핑 (부동 소수점 오차 방지)
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        width = max(0.0, min(1.0, width))
        height = max(0.0, min(1.0, height))

        bbox_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return "\n".join(bbox_lines)

def process_remaining_exception_files():
    """
    메인 라벨링 스크립트 실행 후에도 라벨이 생성되지 않은 JSON 파일들을 찾아
    파일명 기반 오버라이드 로직을 사용하여 라벨을 재처리/생성합니다.
    """
    print("\n" + "="*70)
    print("Starting processing for remaining unhandled JSON files...")
    print("  (Will only process JSONs that currently lack a corresponding .txt label)")
    print("  (Using filename-based class override for these specific files)")
    print("="*70 + "\n")

    reprocessed_count = 0
    skipped_count = 0
    
    # YOLO labels 폴더에 이미 생성된 라벨 파일 목록을 수집합니다.
    existing_labels = set()
    for subset in ['train', 'val', 'test']:
        label_dir = yolo_root / "labels" / subset
        if label_dir.exists():
            for label_file in label_dir.glob("*.txt"):
                existing_labels.add(label_file.stem) # 확장자 제외한 파일명만 저장

    print(f"Found {len(existing_labels)} existing label files across train/val/test.")

    # 모든 원본 JSON 파일 목록을 수집합니다 (Training, Validation).
    all_origin_json_files = []
    for dataset_type in ["Training", "Validation"]:
        dataset_path = origin_root / dataset_type
        if dataset_path.exists():
            for folder in dataset_path.iterdir():
                if folder.is_dir() and folder.name.startswith("[라벨]"):
                    all_origin_json_files.extend(list(folder.glob("*.json")))
    
    if not all_origin_json_files:
        print("No JSON files found in 'origin_sample_dataset/Training' or 'Validation'. Exiting.")
        return

    print(f"Found {len(all_origin_json_files)} total JSON files in the original dataset.")

    # 아직 라벨이 없는 JSON 파일만 필터링합니다.
    unprocessed_json_files = []
    for json_file in all_origin_json_files:
        # JSON 파일명에서 .jpg, .JPG 등 이미지 확장자 제거하여 비교
        clean_stem = json_file.stem.replace(".jpg", "").replace(".JPG", "").replace(".png", "").replace(".PNG", "")
        if clean_stem not in existing_labels:
            unprocessed_json_files.append(json_file)
    
    if not unprocessed_json_files:
        print("\nAll JSON files already have corresponding labels. No further processing needed.")
        return
    
    print(f"\nIdentified {len(unprocessed_json_files)} JSON files that are missing labels.")
    print("Attempting to generate labels for these files now...\n")

    # 미처리된 JSON 파일들을 대상으로 라벨 생성 시도
    for json_file in tqdm(unprocessed_json_files, desc="  Processing missing labels"):
        img_filename_stem = json_file.stem.replace(".jpg", "").replace(".JPG", "").replace(".png", "").replace(".PNG", "")
        
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            # JSON 파일에서 이미지 너비/높이 추출. 없으면 건너뜀.
            img_width = json_data['description']['width']
            img_height = json_data['description']['height']

            # JSON 파일이 어느 원본 폴더에서 왔는지 확인하여 target_type 결정
            target_type = None
            if "Training" in str(json_file.parent):
                target_type = "train"
            elif "Validation" in str(json_file.parent):
                # 여기서는 메인 스크립트의 분할 로직을 정확히 재현하기 어렵기 때문에
                # 누락된 파일을 일괄적으로 'val'에 할당합니다.
                # 필요시 이 부분을 더 정교하게 제어할 수 있습니다.
                target_type = "val" 
            
            if target_type is None:
                # print(f"    ! Warning: Could not determine target subset for {json_file.name}. Skipping.")
                skipped_count += 1
                continue

            label_dir = yolo_root / "labels" / target_type
            txt_path = label_dir / f"{img_filename_stem}.txt"

            # 다시 한번 존재 여부 확인 (경로 유추 후)
            if txt_path.exists():
                # print(f"    Info: Label {txt_path.name} already exists after re-check. Skipping re-creation.")
                continue

            # 파일명 기반으로 라벨 변환 함수 호출
            yolo_label = convert_label_from_filename_override(json_file, img_filename_stem, img_width, img_height)

            if yolo_label:
                label_dir.mkdir(parents=True, exist_ok=True) # 혹시라도 폴더가 없는 경우 대비
                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(yolo_label)
                reprocessed_count += 1
                # print(f"    ✔ Successfully generated label for {json_file.name} in {target_type} folder using filename override.")
            else:
                skipped_count += 1
                # print(f"    ! Failed to generate label for {json_file.name} (no valid YOLO labels produced even with filename override).")

        except Exception as e:
            skipped_count += 1
            # print(f"  ! Error processing {json_file.name}: {e}. Skipping.")

    print(f"\nProcessing of remaining files completed.")
    print(f"  - Successfully generated labels: {reprocessed_count}")
    print(f"  - Skipped (e.g., still problematic or target unknown): {skipped_count}")
    print("="*70 + "\n")

# 이 함수를 메인 라벨링 스크립트 실행 후에 호출하세요.
print("\n--- Starting exception file processing ---")
process_remaining_exception_files()
print("--- Exception file processing finished ---")

# 최종 통계 출력 (모든 처리가 끝난 후)
def print_final_stats():
    print("\n---")
    print("Final Dataset Split Statistics (Labels Only):")
    for split in ['train', 'val', 'test']:
        label_count = len(list((yolo_root / "labels" / split).glob("*.txt")))
        print(f"  - {split}: {label_count} labels")
    print("---\n")

print_final_stats()


--- Starting exception file processing ---

Starting processing for remaining unhandled JSON files...
  (Will only process JSONs that currently lack a corresponding .txt label)
  (Using filename-based class override for these specific files)

Found 67522 existing label files across train/val/test.
Found 67526 total JSON files in the original dataset.

Identified 4 JSON files that are missing labels.
Attempting to generate labels for these files now...



  Processing missing labels: 100%|██████████| 4/4 [00:00<00:00, 668.57it/s]


Processing of remaining files completed.
  - Successfully generated labels: 4
  - Skipped (e.g., still problematic or target unknown): 0

--- Exception file processing finished ---

---
Final Dataset Split Statistics (Labels Only):


  - train: 60015 labels
  - val: 3757 labels
  - test: 3754 labels
---



In [12]:
#===================================================================================================
# 라벨에 대응하는 이미지를 origin 폴더에서 복사해서 images에 넣기
#===================================================================================================
import os
import shutil
from pathlib import Path
from tqdm.auto import tqdm

# 경로 설정
base_path = Path("C:/Users/smhrd/Desktop/데이터셋 생성")
labels_test_dir = base_path / "YOLOv11Dataset/labels/test"
labels_val_dir = base_path / "YOLOv11Dataset/labels/val"
origin_validation_dir = base_path / "origin_sample_dataset/Validation"
images_test_dir = base_path / "YOLOv11Dataset/images/test"
images_val_dir = base_path / "YOLOv11Dataset/images/val"

# 이미지 디렉토리 생성
images_test_dir.mkdir(parents=True, exist_ok=True)
images_val_dir.mkdir(parents=True, exist_ok=True)

# [1] 테스트/검증 라벨 파일명에서 '.txt' 제거 → 실제 이미지 파일명으로 리스트 생성
test_image_filenames = [f.name.replace('.txt', '') for f in labels_test_dir.glob("*.txt")]
val_image_filenames = [f.name.replace('.txt', '') for f in labels_val_dir.glob("*.txt")]

print(f"[INFO] test 이미지 수: {len(test_image_filenames)}")
print(f"[INFO] val 이미지 수:  {len(val_image_filenames)}")

# 복사된 수를 세기 위한 카운터
copied_test = 0
copied_val = 0
not_found = 0

# [2] [원천] 폴더 내 모든 이미지 검색 및 이동 (진행 표시)
all_image_files = []

for folder in origin_validation_dir.iterdir():
    if folder.is_dir() and "[원천]" in folder.name:
        all_image_files.extend([
            f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

print(f"[INFO] 총 원천 이미지 파일 수: {len(all_image_files)}\n")

# 진행 바 표시
for image_file in tqdm(all_image_files, desc="이미지 매칭 및 복사 진행 중"):
    image_name = image_file.name

    if image_name in test_image_filenames:
        shutil.copy2(image_file, images_test_dir / image_name)
        copied_test += 1
        # print(f"[TEST] Copied: {image_name}")
    elif image_name in val_image_filenames:
        shutil.copy2(image_file, images_val_dir / image_name)
        copied_val += 1
        # print(f"[VAL]  Copied: {image_name}")
    else:
        not_found += 1
        # print(f"[WARN] Label not found for image: {image_name}")

# 결과 출력
print("\n[SUMMARY]")
print(f"Copied to test: {copied_test}")
print(f"Copied to val:  {copied_val}")
print(f"Images not matched to any label: {not_found}")


c:\Users\smhrd\Anaconda3\envs\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] test 이미지 수: 3755
[INFO] val 이미지 수:  3756
[INFO] 총 원천 이미지 파일 수: 7511



이미지 매칭 및 복사 진행 중: 100%|██████████| 7511/7511 [01:52<00:00, 66.77it/s] 


[SUMMARY]
Copied to test: 3755
Copied to val:  3756
Images not matched to any label: 0


In [8]:

#===================================================================================================
# 폴더 비교코드
# #===================================================================================================

from pathlib import Path

# 경로 설정
base_path = Path("C:/Users/smhrd/Desktop/데이터셋 생성")
labels_dir = base_path / "YOLOv11Dataset/labels"
images_dir = base_path / "YOLOv11Dataset/images"

# 비교 함수 정의
def check_label_image_match(subset: str):
    """
    주어진 subset ('val' 또는 'test')에 대해 라벨과 이미지 파일명의 일치 여부를 검증합니다.
    """
    print(f"\n[CHECK] {subset.upper()} 이미지와 라벨 파일명 일치 여부 확인 중...")

    label_files = list((labels_dir / subset).glob("*.txt"))
    image_files = list((images_dir / subset).glob("*"))

    # 라벨과 이미지의 파일명(stem)만 추출하여 집합 생성
    label_stems = set(f.stem for f in label_files)
    image_stems = set(f.stem for f in image_files)

    only_in_labels = label_stems - image_stems
    only_in_images = image_stems - label_stems
    matched = label_stems & image_stems

    print(f"  - 총 라벨 수:   {len(label_stems)}")
    print(f"  - 총 이미지 수: {len(image_stems)}")
    print(f"  - 일치 항목 수: {len(matched)}")

    if only_in_labels:
        print(f"  - [!] 이미지가 없는 라벨 파일명 {len(only_in_labels)}개:")
        for name in sorted(only_in_labels)[:10]:
            print(f"      - {name}.txt")
        if len(only_in_labels) > 10:
            print("      ...")

    if only_in_images:
        print(f"  - [!] 라벨이 없는 이미지 파일명 {len(only_in_images)}개:")
        for name in sorted(only_in_images)[:10]:
            print(f"      - {name}.[jpg/jpeg/png]")
        if len(only_in_images) > 10:
            print("      ...")

    if not only_in_labels and not only_in_images:
        print("  ✅ 모든 라벨과 이미지가 정확히 일치합니다.")

# 실행
check_label_image_match("val")
check_label_image_match("test")



[CHECK] VAL 이미지와 라벨 파일명 일치 여부 확인 중...
  - 총 라벨 수:   3757
  - 총 이미지 수: 3756
  - 일치 항목 수: 1842
  - [!] 이미지가 없는 라벨 파일명 1915개:
      - V006_80_0_00_01_01_25_0_b07_20201005_0001_S01_1.txt
      - V006_80_0_00_01_01_25_0_b07_20201005_0005_S01_1.txt
      - V006_80_0_00_01_03_25_0_b06_20201013_0016_S01_1.txt
      - V006_80_0_00_01_03_25_0_b06_20201013_0018_S01_1.txt
      - V006_80_0_00_01_03_25_0_b06_20201021_0000_S01_1.txt
      - V006_80_0_00_01_03_25_0_b06_20201028_0019_S01_1.txt
      - V006_80_0_00_01_03_25_0_b06_20201028_0040_S01_1.txt
      - V006_80_0_00_01_03_25_0_b07_20201005_0011_S01_1.txt
      - V006_80_0_00_01_03_25_0_b07_20201105_0021_S01_1.txt
      - V006_80_0_00_01_03_25_0_b07_20201105_0039_S01_1.txt
      ...
  - [!] 라벨이 없는 이미지 파일명 1914개:
      - V006_80_0_00_01_01_25_0_b06_20201005_0011_S01_1.[jpg/jpeg/png]
      - V006_80_0_00_01_01_25_0_b07_20201005_0009_S01_1.[jpg/jpeg/png]
      - V006_80_0_00_01_01_25_0_b07_20201012_0002_S01_1.[jpg/jpeg/png]
      - V006_80_0_00_01

In [ ]:
#=================================
# 이미지 기준으로 라벨 재배치
#=================================
import os
from pathlib import Path
import shutil

# 경로 설정
base_path = Path("C:/Users/smhrd/Desktop/데이터셋 생성")
labels_dir = base_path / "YOLOv11Dataset/labels"
images_dir = base_path / "YOLOv11Dataset/images"

# 지원되는 이미지 확장자 목록 (필요에 따라 추가)
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif", ".JPG", ".JPEG", ".PNG", ".BMP", ".TIFF", ".GIF"]

def synchronize_labels_with_images(subset: str):
    """
    주어진 subset ('val' 또는 'test')에 대해 라벨 파일을 이미지 파일에 맞춰 동기화합니다.
    - images/subset에 있는 이미지 파일에 해당하는 라벨 파일만 labels/subset에 남깁니다.
    - images/subset에 없는 라벨 파일 (labels/subset에만 있는 파일)은 삭제합니다.
    - images/subset에 있으나 labels/subset에 없는 라벨 파일은 이 스크립트에서 생성하지 않습니다.
      (생성하려면 원본 JSON 데이터 및 변환 로직이 필요하기 때문입니다.)
    """
    print(f"\n--- {subset.upper()} 라벨-이미지 동기화 시작 ---")

    subset_labels_dir = labels_dir / subset
    subset_images_dir = images_dir / subset

    if not subset_labels_dir.exists():
        print(f"  경고: 라벨 디렉토리 '{subset_labels_dir}'가 존재하지 않습니다. 스킵합니다.")
        return
    if not subset_images_dir.exists():
        print(f"  경고: 이미지 디렉토리 '{subset_images_dir}'가 존재하지 않습니다. 스킵합니다.")
        return

    # 1. 이미지 파일의 stem (확장자 없는 이름) 목록 가져오기
    image_stems = set()
    for ext in IMAGE_EXTENSIONS:
        for img_file in subset_images_dir.glob(f"*{ext}"):
            image_stems.add(img_file.stem)
    
    print(f"  {subset.upper()} 이미지 파일 수: {len(image_stems)}")

    # 2. 현재 라벨 파일의 stem 목록 가져오기
    current_label_stems = {f.stem for f in subset_labels_dir.glob("*.txt")}
    print(f"  {subset.upper()} 현재 라벨 파일 수: {len(current_label_stems)}")

    # 3. 이미지에는 없지만 라벨에만 있는 파일 찾기 (삭제 대상)
    labels_to_remove = current_label_stems - image_stems
    removed_count = 0

    if labels_to_remove:
        print(f"  [삭제 대상] 이미지에 해당하지 않는 라벨 파일 {len(labels_to_remove)}개 발견:")
        for stem in sorted(list(labels_to_remove))[:10]:
            print(f"    - {stem}.txt")
        if len(labels_to_remove) > 10:
            print("    ...")

        confirm = input(f"  위 라벨 파일들을 정말 삭제하시겠습니까? (y/n): ")
        if confirm.lower() == 'y':
            for stem in labels_to_remove:
                label_file_path = subset_labels_dir / f"{stem}.txt"
                try:
                    os.remove(label_file_path)
                    removed_count += 1
                except OSError as e:
                    print(f"  오류: 라벨 파일 삭제 실패 '{label_file_path}': {e}")
            print(f"  총 {removed_count}개의 라벨 파일 삭제 완료.")
        else:
            print("  삭제 작업을 취소했습니다.")
    else:
        print("  삭제할 라벨 파일이 없습니다. 모든 라벨이 이미지에 해당합니다.")

    # 4. 이미지에는 있지만 라벨이 없는 파일 목록 (정보 제공용)
    # 이 스크립트에서는 생성하지 않습니다.
    missing_labels_for_images = image_stems - current_label_stems
    if missing_labels_for_images:
        print(f"  [정보] 라벨이 없는 이미지 파일 {len(missing_labels_for_images)}개 발견:")
        for stem in sorted(list(missing_labels_for_images))[:10]:
            # 실제 이미지 확장자를 포함하여 출력
            found_img_path = None
            for ext in IMAGE_EXTENSIONS:
                if (subset_images_dir / f"{stem}{ext}").exists():
                    found_img_path = subset_images_dir / f"{stem}{ext}"
                    break
            if found_img_path:
                print(f"    - {found_img_path.name}")
            else:
                print(f"    - {stem}.(확장자 알 수 없음)") # 안전 장치
        if len(missing_labels_for_images) > 10:
            print("    ...")
        print("  참고: 이 스크립트는 누락된 라벨 파일을 새로 생성하지 않습니다. 원본 JSON 데이터를 사용하여 별도로 생성해야 합니다.")
    else:
        print("  누락된 라벨 파일이 없습니다. 모든 이미지가 라벨에 해당합니다.")

    # 최종 상태 확인
    final_label_stems = {f.stem for f in subset_labels_dir.glob("*.txt")}
    
    if final_label_stems == image_stems:
        print(f"  ✅ {subset.upper()} 라벨과 이미지가 성공적으로 동기화되었습니다. (라벨: {len(final_label_stems)}개, 이미지: {len(image_stems)}개)")
    else:
        print(f"  ❗ {subset.upper()} 동기화 후에도 불일치가 남아있습니다. 추가 조치가 필요할 수 있습니다.")
        print(f"    - 라벨만 있는 파일 (삭제되지 않은): {len(final_label_stems - image_stems)}")
        print(f"    - 이미지만 있는 파일 (라벨이 없는): {len(image_stems - final_label_stems)}")

    print(f"--- {subset.upper()} 라벨-이미지 동기화 완료 ---\n")


# 스크립트 실행
if __name__ == "__main__":
    print("YOLOv11Dataset 라벨-이미지 파일 동기화 스크립트 시작")
    
    synchronize_labels_with_images("val")
    synchronize_labels_with_images("test")
    # train 폴더도 동기화하려면 다음 라인의 주석을 해제하세요.
    # synchronize_labels_with_images("train") 
    
    print("YOLOv11Dataset 라벨-이미지 파일 동기화 스크립트 완료")